# MAS-IDS — Kaggle Runner
## Multi-Agent IDS for DoS/DDoS & Jamming (UAV/UGV networks)

This notebook pulls the **modular `mas_ids` package** from your GitHub repo and
runs the full 6-agent pipeline. The heavy code lives in clean, debuggable `.py`
modules — this notebook is just the driver.

**Setup once:** edit `GITHUB_USER` / `REPO` / `BRANCH` below to point at your repo.

> Make sure Kaggle **Internet** is ON (Settings → Internet) so the clone works.

## 1 — Clone the repo from GitHub

In [ ]:
# ── EDIT THESE THREE LINES to match your GitHub repo ─────────────────────────
GITHUB_USER = 'YOUR_GITHUB_USERNAME'
REPO        = 'mas-ids'
BRANCH      = 'main'
# ──────────────────────────────────────────────────────────────────────────────

import os, shutil, sys

REPO_URL = f'https://github.com/{GITHUB_USER}/{REPO}.git'
CLONE_DIR = f'/kaggle/working/{REPO}'

# Fresh clone each run (so you always get the latest pushed code)
if os.path.exists(CLONE_DIR):
    shutil.rmtree(CLONE_DIR)
!git clone --branch {BRANCH} --depth 1 {REPO_URL} {CLONE_DIR}

# Put the repo on the import path
if CLONE_DIR not in sys.path:
    sys.path.insert(0, CLONE_DIR)
print('Repo on path:', CLONE_DIR)

### Alternative: add the repo as a Kaggle *Dataset* or *Utility Script*
If you prefer not to use Internet access, you can instead:
1. Push the repo to GitHub, then on Kaggle: **Add Input → GitHub** and link it, **or**
2. Upload the `mas_ids/` folder as a Kaggle Dataset and set
   `CLONE_DIR = '/kaggle/input/<your-dataset-slug>'`.

Either way, the only requirement is that `CLONE_DIR` ends up on `sys.path`.

## 2 — Environment setup (seeds, GPU, output dirs)

## 2b — Choose your dataset source

Run **one** of the two blocks below, then continue to Section 3.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# OPTION A — Synthetic data (no upload required, good for quick tests)
# ══════════════════════════════════════════════════════════════════════
USE_REAL_DATASET = False   # ← set True to use UAV-NIDD instead

if not USE_REAL_DATASET:
    cleaned_df = edge_df = gcc_df = all_df = None   # pipeline generates internally
    print('Using synthetic TrafficDataGenerator data.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# OPTION B — UAV-NIDD real dataset (recommended for research)
#
# 1. Download from https://zenodo.org/records/15125851
# 2. On Kaggle: + Add Data → Upload Dataset → upload UAV-NDD CSV.zip
#    (Kaggle will auto-unzip; the CSVs will appear under /kaggle/input/)
# 3. Set the path below to match your dataset slug.
# ══════════════════════════════════════════════════════════════════════
USE_REAL_DATASET = True

if USE_REAL_DATASET:
    from mas_ids.data.uav_nidd_loader import load_uav_nidd_split

    UAV_NIDD_DIR = '/kaggle/input/uav-nidd/'   # ← adjust to your dataset path

    cleaned_df, edge_df, gcc_df, all_df = load_uav_nidd_split(
        csv_dir        = UAV_NIDD_DIR,
        label_col      = 'Label',
        window_seconds = 1.0,
        max_rows       = None,     # set e.g. 500_000 for a quick test run
        verbose        = True,
    )
    print(f'cleaned_df: {cleaned_df.shape}')
    print(cleaned_df['label'].value_counts().to_string())

In [ ]:
from mas_ids.config import setup_environment
import mas_ids.config as cfg

setup_environment()           # seeds + GPU memory growth + mixed precision + dirs

# Optional: shrink the run for a quick smoke test (comment out for full run)
# cfg.TRAIN_EPOCHS = 5
# cfg.DQN_EPISODES = 20
print('TRAIN_EPOCHS =', cfg.TRAIN_EPOCHS, '| DQN_EPISODES =', cfg.DQN_EPISODES)

## 3 — Run the full pipeline (all 6 agents)

`run_full_mas_pipeline()` runs Data → Features → Detection → Response →
Coordination/Management → Logger and returns every intermediate DataFrame.

In [ ]:
from mas_ids.pipeline import run_full_mas_pipeline

# If USE_REAL_DATASET=True, pass cleaned_df so the pipeline skips generation
pipeline_kwargs = dict(preview=True)
if 'USE_REAL_DATASET' in dir() and USE_REAL_DATASET and cleaned_df is not None:
    # run_full_mas_pipeline accepts a pre-built cleaned_df
    pipeline_kwargs['cleaned_df'] = cleaned_df

results = run_full_mas_pipeline(**pipeline_kwargs)

detection_df = results['detection_df']
response_df  = results['response_df']
coord_df     = results['coord_df']
mgmt_df      = results['mgmt_df']
logger       = results['logger']
eval_results = results['eval_results']

print('\nDetection F1 :', round(eval_results.get('f1', 0), 4))
print('Logger events:', len(logger.logs))
print('Integrity    :', results['integrity']['status'])

## 4 — Run agents individually (for debugging / experiments)

Because each agent is its own module, you can import and run them in isolation
to debug a single stage without re-running the whole pipeline.

In [ ]:
# Example: run only Data + Feature engineering, inspect the feature matrix
from mas_ids.agents.data_agent import (
    TrafficDataGenerator, EdgeCollector, GCCCollector,
    DataQualityChecker, DataCleaner,
)
from mas_ids.config import N_NORMAL, N_JAMMING, N_DOS, N_DDOS, N_HYBRID, GLOBAL_SEED
import pandas as pd

gen   = TrafficDataGenerator(seed=GLOBAL_SEED)
raw   = gen.generate(n_normal=200, n_jamming=80, n_dos=60, n_ddos=60, n_hybrid=40)
print('raw shape:', raw.shape)
print(raw['label'].value_counts().to_string())

## 5 — Single-window inference (deployment-style API)

Each agent ships a `*_single()` helper for one-shot inference on a hand-crafted
window — useful for unit tests or live deployment.

In [ ]:
from mas_ids.agents.detection_agent import predict_single

# Uses the models + fe_state already trained in step 3
det_models = results.get('det_models')
fe_state   = results['fe_state']

# A jamming-like window
jam_window = {
    'rssi_dbm':-108.0,'snr_db':-5.0,'sinr_db':-8.0,'noise_floor_dbm':-65.0,
    'bit_error_rate':0.28,'bad_packet_ratio':0.42,'packet_delivery_ratio':0.38,
    'mac_retry_count':120,'channel_occupancy_pct':88.0,'packets_per_second':45.0,
    'src_ip_entropy':2.1,'syn_packet_rate':1.5,
}
if det_models is not None:
    out = predict_single(jam_window, det_models, fe_state, preview=True)
else:
    print('Re-run step 3 first to train det_models, or load them from disk.')

## 6 — Outputs

All artefacts are written to `/kaggle/working/`:
- `saved_detection_models/` — trained `.keras` models + config
- `saved_response_models/`  — DQN policy weights
- `mas_ids_unified_logs.csv` / `.jsonl` — tamper-evident event log
- `mas_ids_unified_merkle_batches.csv` — Merkle integrity batches

Download them from the Kaggle **Output** tab, or read them back below.

In [ ]:
import os
for f in sorted(os.listdir('/kaggle/working')):
    p = os.path.join('/kaggle/working', f)
    if os.path.isfile(p):
        print(f'{f:<45} {os.path.getsize(p)//1024:>6} KB')